In [ ]:
# pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [5]:
pip install --upgrade transformers sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
chunks = [
    {
        "id": "chunk_001",
        "text": "Python is a programming language used for machine learning.",
        "topic": "Python"
    },
    {
        "id": "chunk_002",
        "text": "PyTorch is a framework used to build and train deep learning models.",
        "topic": "PyTorch"
    },
    {
        "id": "chunk_003",
        "text": "Neo4j stores data as nodes and relationships in a graph.",
        "topic": "Neo4j"
    }
] 

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

for chunk in chunks:
    vector = model.encode(
        chunk["text"],
        normalize_embeddings=True
    )

    chunk["embedding"] = vector.tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
from neo4j import GraphDatabase

# Neo4j Desktop default URI is bolt://localhost:7687
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "12345678")  # Replace with your actual password

driver = GraphDatabase.driver(URI, auth=AUTH)

In [12]:
# use unwind for batch ingestion 
def insert_chunk(tx,chunks):
    result = tx.run(
        """
        UNWIND $chunks AS item
        MERGE (c:Chunk{id:item.id})
        SET c.text = item.text,
        c.topic = item.topic,
        c.embedding = item.embedding
        RETURN count(c) AS processed
        """,
        chunks = chunks
    )
    return result.single()['processed']

In [13]:
with driver.session() as session:
    count = session.execute_write(
        insert_chunk,
        chunks
    )
    print("Processed:",count)

Processed: 3


In [14]:
## 9.8 # query the vector index
question = "which tool help build deep learning models ?"

query_vector = model.encode(
    question,
    normalize_embeddings=True
)

In [ ]:
qu